In [1]:
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from import_folder import import_all_noiselevel_data
from sklearn.model_selection import train_test_split

# -----------------------------------------------------------------------------
# Data Load and Definitions
# -----------------------------------------------------------------------------
folders = ['SNR_noiseless', 'SNR_50dB', 'SNR_40dB', 'SNR_30dB', 'SNR_20dB']

# Load all noise levels into a single DataFrame
all_data = import_all_noiselevel_data(folders, add_noise_column=False)

# Separate Features and Label
X = all_data.drop(columns=['label'])
y = all_data['label']
# Define cv strategy for hyperparameter tuning
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring metric for hyperparameter tuning
scoring_metric = 'f1_macro'

# random search parameters
num_combinations = 20  # Number of hyperparameter sets sampled

# other definitions
RANDOM_STATE = 42

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [2]:
# -----------------------------------------------------------------------------
# Random Forest (RF) Randomized Search
# -----------------------------------------------------------------------------
rf_param_dist = {
    "classifier__n_estimators": randint(50, 500),
    "classifier__max_depth": randint(4, 25),
    "classifier__min_samples_split": randint(2, 20),
    "classifier__max_features": ["sqrt", "log2", None],
}

rf_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ]
)

rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=num_combinations,
    scoring=scoring_metric,
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# -----------------------------------------------------------------------------
# Fit the Randomized Searches on Training Data
# -----------------------------------------------------------------------------
print("Fitting Random Forest Randomized Search...")
rf_random.fit(X_train, y_train)

print("\n=== TUNING COMPLETE ===")
print(f"Best RF CV {scoring_metric}: {rf_random.best_score_:.4f}")

# -----------------------------------------------------------------------------
# Inspect Best Hyperparameters Found by Randomized Search
# -----------------------------------------------------------------------------
print("=== BEST HYPERPARAMETERS FOUND ===")

print("\n2. Random Forest (RF):")
for param, value in rf_random.best_params_.items():
    print(f"   {param}: {value}")

Fitting Random Forest Randomized Search...

=== TUNING COMPLETE ===
Best RF CV f1_macro: 0.9847
=== BEST HYPERPARAMETERS FOUND ===

2. Random Forest (RF):
   classifier__max_depth: 11
   classifier__max_features: sqrt
   classifier__min_samples_split: 8
   classifier__n_estimators: 171
